## Vector RTL-Coder Dataset Verification

In [2]:
import pandas as pd

file_path = "vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"

df = pd.read_parquet(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\mikel\AppData\Local\Temp\ipykernel_12852\3858332521.py:3: SyntaxWarning: invalid escape sequence '\p'
  file_path = "vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"


Shape: (26532, 4)

Columns:
['Instruction', 'VerilogCode', 'id', 'embedding_text']

Data types:
Instruction       object
VerilogCode       object
id                 int64
embedding_text    object
dtype: object

First 5 rows:
                                         Instruction  \
0  \nYou are tasked with designing a module for a...   
1  Design a module that can detect any edge in an...   
2  Please act as a professional Verilog designer....   
3  Create a new module that combines the function...   
4  Please act as a professional Verilog designer....   

                                         VerilogCode  id  \
0  module calculator(\n    input logic [31:0] a,\...   0   
1  module edge_detector (\n    input clk,\n    in...   1   
2  module priority_encoder (\n    input [3:0] I,\...   2   
3  module shift_reg (\n    input clk,\n    input ...   3   
4  module clock_gating_cell (\n  input clk,\n  in...   4   

                                      embedding_text  
0  \nYou are tasked wi

In [3]:
import pandas as pd

file_path = r"vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"

df = pd.read_parquet(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

# Fill missing values to avoid errors
df["Instruction"] = df["Instruction"].fillna("")
df["VerilogCode"] = df["VerilogCode"].fillna("")

# Character counts
df["instruction_chars"] = df["Instruction"].str.len()
df["verilog_chars"] = df["VerilogCode"].str.len()
df["question_code_chars"] = df["instruction_chars"] + df["verilog_chars"]

# Max row
max_idx = df["question_code_chars"].idxmax()
max_row = df.loc[max_idx]

print("\nMax characters for Question + Code:")
print("Row index:", max_idx)
print("id:", max_row["id"])
print("Instruction chars:", max_row["instruction_chars"])
print("Verilog chars:", max_row["verilog_chars"])
print("Total chars:", max_row["question_code_chars"])

print("\nSummary stats for Question + Code chars:")
print(df["question_code_chars"].describe())

print("\nTop 5 longest Question + Code rows:")
print(
    df[["id", "instruction_chars", "verilog_chars", "question_code_chars"]]
    .sort_values("question_code_chars", ascending=False)
    .head(5)
)

print("\nLongest Instruction preview:")
print(max_row["Instruction"][:500])

print("\nLongest VerilogCode preview:")
print(max_row["VerilogCode"][:500])

Shape: (26532, 4)

Columns:
['Instruction', 'VerilogCode', 'id', 'embedding_text']

Data types:
Instruction       object
VerilogCode       object
id                 int64
embedding_text    object
dtype: object

First 5 rows:
                                         Instruction  \
0  \nYou are tasked with designing a module for a...   
1  Design a module that can detect any edge in an...   
2  Please act as a professional Verilog designer....   
3  Create a new module that combines the function...   
4  Please act as a professional Verilog designer....   

                                         VerilogCode  id  \
0  module calculator(\n    input logic [31:0] a,\...   0   
1  module edge_detector (\n    input clk,\n    in...   1   
2  module priority_encoder (\n    input [3:0] I,\...   2   
3  module shift_reg (\n    input clk,\n    input ...   3   
4  module clock_gating_cell (\n  input clk,\n  in...   4   

                                      embedding_text  
0  \nYou are tasked wi

## VeriRAG Dataset Preparation

In [1]:
import pandas as pd

file_path = "vector_csv_VERIRAG\VeriRAG_Dataset.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

<>:3: SyntaxWarning: invalid escape sequence '\V'
<>:3: SyntaxWarning: invalid escape sequence '\V'
C:\Users\mikel\AppData\Local\Temp\ipykernel_28772\303640037.py:3: SyntaxWarning: invalid escape sequence '\V'
  file_path = "vector_csv_VERIRAG\VeriRAG_Dataset.csv"


Shape: (161, 1)

Columns:
['text']

Data types:
text    object
dtype: object

First 5 rows:
                                                text
0  <s>[INST] Write Verilog code for a 2-input AND...
1  <s>[INST] Write Verilog code for a 2-input OR ...
2  <s>[INST] Write Verilog code for a NOT gate. [...
3  <s>[INST] Write Verilog code for a 2-input NAN...
4  <s>[INST] Write Verilog code for a 2-input NOR...


In [2]:
import re
import pandas as pd
from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
file_path = Path("vector_csv_VERIRAG/VeriRAG_Dataset.csv")
output_parquet = Path("vector_parquet/verirag_clean.parquet")

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

# -----------------------------
# Parser
# Expected format:
# <s>[INST] instruction [/INST] verilog </s>
# -----------------------------
pattern = re.compile(
    r"<s>\s*\[INST\]\s*(.*?)\s*\[/INST\]\s*(.*?)\s*</s>\s*$",
    re.DOTALL
)

def parse_text(row_text: str):
    if pd.isna(row_text):
        return None, None, "missing_text"
    
    text = str(row_text).strip()
    match = pattern.match(text)
    
    if not match:
        return None, None, "parse_failed"
    
    instruction = match.group(1).strip()
    verilog = match.group(2).strip()
    
    # Optional cleanup of repeated whitespace
    instruction = re.sub(r"\s+", " ", instruction).strip()
    verilog = verilog.strip()
    
    return instruction, verilog, None

parsed = df["text"].apply(parse_text)

df["Instruction"] = parsed.apply(lambda x: x[0])
df["VerilogCode"] = parsed.apply(lambda x: x[1])
df["parse_error"] = parsed.apply(lambda x: x[2])

# -----------------------------
# Inspect parsing quality
# -----------------------------
print("\nParse error counts:")
print(df["parse_error"].value_counts(dropna=False))

bad_rows = df[df["parse_error"].notna()]
if not bad_rows.empty:
    print("\nSample bad rows:")
    print(bad_rows[["text", "parse_error"]].head(10))

# -----------------------------
# Keep only valid rows
# -----------------------------
clean_df = df[df["parse_error"].isna()].copy()

# Basic validation
clean_df["has_module_keyword"] = clean_df["VerilogCode"].str.contains(r"\bmodule\b", regex=True, na=False)
clean_df["has_endmodule_keyword"] = clean_df["VerilogCode"].str.contains(r"\bendmodule\b", regex=True, na=False)

print("\nRows without 'module':", (~clean_df["has_module_keyword"]).sum())
print("Rows without 'endmodule':", (~clean_df["has_endmodule_keyword"]).sum())

# Optional: inspect suspicious rows
suspicious = clean_df[
    (~clean_df["has_module_keyword"]) | (~clean_df["has_endmodule_keyword"])
]
if not suspicious.empty:
    print("\nSuspicious parsed rows:")
    print(suspicious[["Instruction", "VerilogCode"]].head(10))

# -----------------------------
# Final schema to match old parquet
# -----------------------------
clean_df = clean_df.reset_index(drop=True)
clean_df["id"] = clean_df.index
clean_df["embedding_text"] = clean_df["Instruction"]

final_df = clean_df[["Instruction", "VerilogCode", "id", "embedding_text"]].copy()

print("\nFinal shape:", final_df.shape)
print("\nFinal columns:", final_df.columns.tolist())
print("\nSample final rows:")
print(final_df.head())

# -----------------------------
# Save parquet
# -----------------------------
output_parquet.parent.mkdir(parents=True, exist_ok=True)
final_df.to_parquet(output_parquet, index=False)

print(f"\nSaved parquet to: {output_parquet}")

Original shape: (161, 1)
Columns: ['text']
                                                text
0  <s>[INST] Write Verilog code for a 2-input AND...
1  <s>[INST] Write Verilog code for a 2-input OR ...
2  <s>[INST] Write Verilog code for a NOT gate. [...
3  <s>[INST] Write Verilog code for a 2-input NAN...
4  <s>[INST] Write Verilog code for a 2-input NOR...

Parse error counts:
parse_error
None    161
Name: count, dtype: int64

Rows without 'module': 0
Rows without 'endmodule': 0

Final shape: (161, 4)

Final columns: ['Instruction', 'VerilogCode', 'id', 'embedding_text']

Sample final rows:
                                   Instruction  \
0   Write Verilog code for a 2-input AND gate.   
1    Write Verilog code for a 2-input OR gate.   
2           Write Verilog code for a NOT gate.   
3  Write Verilog code for a 2-input NAND gate.   
4   Write Verilog code for a 2-input NOR gate.   

                                         VerilogCode  id  \
0  module and_gate (input wire a, input

In [3]:
import pandas as pd

df = pd.read_parquet("vector_parquet/verirag_clean.parquet")
print(df.shape)
print(df.columns)
print(df.head(3))
print(df.iloc[0]["Instruction"])
print(df.iloc[0]["VerilogCode"])

(161, 4)
Index(['Instruction', 'VerilogCode', 'id', 'embedding_text'], dtype='object')
                                  Instruction  \
0  Write Verilog code for a 2-input AND gate.   
1   Write Verilog code for a 2-input OR gate.   
2          Write Verilog code for a NOT gate.   

                                         VerilogCode  id  \
0  module and_gate (input wire a, input wire b, o...   0   
1  module or_gate (input wire a, input wire b, ou...   1   
2  module not_gate (input wire a, output wire y);...   2   

                               embedding_text  
0  Write Verilog code for a 2-input AND gate.  
1   Write Verilog code for a 2-input OR gate.  
2          Write Verilog code for a NOT gate.  
Write Verilog code for a 2-input AND gate.
module and_gate (input wire a, input wire b, output wire y);
    assign y = a & b;
endmodule


## KG RTL-Coder Dataset & Extraction Pipeline Verification

In [1]:
import pandas as pd

file_path = "kg_features_parquet\part-00000-tid-5475169348745911287-c2bd5a7c-598b-492d-a9d9-f3537b852815-141-1.c000.snappy.parquet"

df = pd.read_parquet(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
C:\Users\mikel\AppData\Local\Temp\ipykernel_25968\3004841789.py:3: SyntaxWarning: invalid escape sequence '\p'
  file_path = "kg_features_parquet\part-00000-tid-5475169348745911287-c2bd5a7c-598b-492d-a9d9-f3537b852815-141-1.c000.snappy.parquet"


Shape: (26532, 49)

Columns:
['doc_id', 'module_name', 'has_always', 'has_assign', 'has_case', 'has_if', 'has_for', 'has_generate', 'has_posedge', 'has_negedge', 'has_clk_signal', 'has_reset_signal', 'tag_async_reset', 'tag_fsm', 'tag_counter', 'tag_mux', 'tag_alu', 'tag_shift', 'tag_mem_array', 'tag_ram', 'tag_fifo', 'has_not_op', 'has_and_op', 'has_or_op', 'has_xor_op', 'has_add_op', 'has_sub_op', 'has_lshift_op', 'has_rshift_op', 'has_concat', 'has_compare_eq', 'has_compare_lt', 'has_compare_gt', 'tag_constant_output', 'tag_not_gate', 'tag_encoder', 'tag_decoder', 'tag_comparator', 'tag_edge_detector', 'tag_byte_reorder', 'has_instantiation', 'tag_wrapper_module', 'num_inputs', 'num_outputs', 'max_decl_width', 'has_wide_port_8plus', 'has_wide_port_32plus', 'is_single_output', 'is_assign_only']

Data types:
doc_id                   int64
module_name             object
has_always               int32
has_assign               int32
has_case                 int32
has_if                  

KG retrieval maps the natural-language specification into a structured hardware feature signature and retrieves examples whose symbolic structural attributes best match that expected design structure.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

PARQUET_PATH = "..."
OUT_PATH = "vector_embeddings_all-MiniLM-L6-v2.npy"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

df = pd.read_parquet(PARQUET_PATH)
df["embedding_text"] = df["embedding_text"].fillna("")

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    df["embedding_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

np.save(OUT_PATH, embeddings)
print("saved:", OUT_PATH, embeddings.shape)

In [ ]:
import re
import pandas as pd

VECTOR_PARQUET_PATH = "vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"
KG_PARQUET_PATH = "kg_features_parquet\part-00000-tid-5475169348745911287-c2bd5a7c-598b-492d-a9d9-f3537b852815-141-1.c000.snappy.parquet"

NUM_TEST_INSTRUCTIONS = 5
TOP_K = 5
MAX_PREVIEW_CHARS = 220


def load_vector_db():
    df = pd.read_parquet(VECTOR_PARQUET_PATH)
    df["Instruction"] = df["Instruction"].fillna("")
    df["VerilogCode"] = df["VerilogCode"].fillna("")
    df["embedding_text"] = df["embedding_text"].fillna("")
    return df


def load_kg_db():
    df = pd.read_parquet(KG_PARQUET_PATH)

    if "module_name" in df.columns:
        df["module_name"] = df["module_name"].fillna("")

    feature_cols = [
        "has_always", "has_assign", "has_case", "has_if", "has_for", "has_generate",
        "has_posedge", "has_negedge", "has_clk_signal", "has_reset_signal",
        "tag_async_reset", "tag_fsm", "tag_counter", "tag_mux", "tag_alu",
        "tag_shift", "tag_mem_array", "tag_ram", "tag_fifo"
    ]

    for col in feature_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    return df


def _normalize_text(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"[_\-/]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _contains_any(text: str, patterns) -> bool:
    for pattern in patterns:
        if re.search(pattern, text):
            return True
    return False


def _add_positive(features: dict, name: str):
    features["positive"][name] = 1


def _add_negative(features: dict, name: str):
    features["negative"][name] = 0


def extract_kg_query_features(prompt: str) -> dict:
    """
    Rule-based prompt-to-feature extractor.
    Returns:
        {
            "positive": {feature_name: 1, ...},
            "negative": {feature_name: 0, ...}
        }
    """
    text = _normalize_text(prompt)

    features = {
        "positive": {},
        "negative": {}
    }

    if not text:
        return features

    # ------------------------------------------------------------------
    # High-confidence functional tags
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bfifo\b", r"\bqueue\b", r"\bread pointer\b", r"\bwrite pointer\b",
        r"\brd_ptr\b", r"\bwr_ptr\b", r"\bread_ptr\b", r"\bwrite_ptr\b"
    ]):
        _add_positive(features, "tag_fifo")
        _add_positive(features, "tag_ram")
        _add_positive(features, "tag_mem_array")
        _add_positive(features, "has_always")
        _add_positive(features, "has_clk_signal")

    if _contains_any(text, [
        r"\bfsm\b", r"\bfinite state machine\b", r"\bstate machine\b",
        r"\bstate transition\b", r"\bnext state\b", r"\bcurrent state\b"
    ]):
        _add_positive(features, "tag_fsm")
        _add_positive(features, "has_case")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\bcounter\b", r"\bup counter\b", r"\bdown counter\b",
        r"\bincrement\b", r"\bdecrement\b", r"\bcounts?\b"
    ]):
        _add_positive(features, "tag_counter")
        _add_positive(features, "has_always")
        _add_positive(features, "has_clk_signal")
        _add_positive(features, "has_posedge")

    if _contains_any(text, [
        r"\bmux\b", r"\bmultiplexer\b", r"\bselector\b", r"\bselect one of\b",
        r"\b4 to 1\b", r"\b2 to 1\b", r"\b8 to 1\b"
    ]):
        _add_positive(features, "tag_mux")

    if _contains_any(text, [
        r"\balu\b", r"\barithmetic logic unit\b", r"\bopcode\b",
        r"\badd/subtract\b", r"\badd and subtract\b"
    ]):
        _add_positive(features, "tag_alu")
        _add_positive(features, "has_case")

    if _contains_any(text, [
        r"\bshift register\b", r"\bshift left\b", r"\bshift right\b",
        r"\bbarrel shifter\b", r"\bshifter\b", r"\bshift\b"
    ]):
        _add_positive(features, "tag_shift")

    if _contains_any(text, [
        r"\bram\b", r"\bmemory\b", r"\bmem\b", r"\bstorage array\b"
    ]):
        _add_positive(features, "tag_ram")
        _add_positive(features, "tag_mem_array")

    if _contains_any(text, [
        r"\balways output[s]?\s+(a\s+)?low\b",
        r"\balways output[s]?\s+(a\s+)?high\b",
        r"\balways drive[s]?\s+0\b",
        r"\balways drive[s]?\s+1\b",
        r"\balways drive[s]?\s+low\b",
        r"\balways drive[s]?\s+high\b",
        r"\bconstant output\b",
        r"\bconstant zero\b",
        r"\bconstant one\b"
    ]):
        _add_positive(features, "tag_constant_output")
        _add_positive(features, "is_assign_only")
        _add_positive(features, "is_single_output")

    if _contains_any(text, [
        r"\bnot gate\b", r"\binverter\b", r"\binvert\b", r"\bbitwise not\b"
    ]):
        _add_positive(features, "tag_not_gate")
        _add_positive(features, "has_not_op")
        _add_positive(features, "is_assign_only")

    if _contains_any(text, [
        r"\bencoder\b", r"\bpriority encoder\b"
    ]):
        _add_positive(features, "tag_encoder")
        _add_positive(features, "has_if")

    if _contains_any(text, [
        r"\bdecoder\b"
    ]):
        _add_positive(features, "tag_decoder")

    if _contains_any(text, [
        r"\bcomparator\b", r"\bcompare\b", r"\bequal\b", r"\bgreater than\b",
        r"\bless than\b", r"\bgreater-than\b", r"\bless-than\b"
    ]):
        _add_positive(features, "tag_comparator")

    if _contains_any(text, [
        r"\bedge detector\b", r"\bdetect any edge\b", r"\brising edge detector\b",
        r"\bfalling edge detector\b", r"\bedge detect\b"
    ]):
        _add_positive(features, "tag_edge_detector")
        _add_positive(features, "has_clk_signal")
        _add_positive(features, "has_posedge")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\breverse the byte order\b", r"\bbyte order\b", r"\bbyte reorder\b",
        r"\bbyte swap\b", r"\bendian\b"
    ]):
        _add_positive(features, "tag_byte_reorder")
        _add_positive(features, "has_concat")
        _add_positive(features, "has_wide_port_32plus")

    if _contains_any(text, [
        r"\binstantiate\b", r"\binstantiates\b", r"\btop level module\b",
        r"\btop-level module\b", r"\bwrapper\b", r"\bsubmodule\b"
    ]):
        _add_positive(features, "has_instantiation")

    # ------------------------------------------------------------------
    # Clock/reset/sequential style
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bclock\b", r"\bclk\b", r"\bsynchronous\b", r"\bposedge\b",
        r"\brising edge\b"
    ]):
        _add_positive(features, "has_clk_signal")
        _add_positive(features, "has_posedge")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\bnegedge\b", r"\bfalling edge\b"
    ]):
        _add_positive(features, "has_negedge")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\breset\b", r"\brst\b"
    ]):
        _add_positive(features, "has_reset_signal")

    if _contains_any(text, [
        r"\basynchronous reset\b", r"\basync reset\b", r"\bactive low reset\b",
        r"\bactive-low reset\b", r"\bnegedge reset\b", r"\bnegedge rst\b"
    ]):
        _add_positive(features, "tag_async_reset")
        _add_positive(features, "has_reset_signal")
        _add_positive(features, "has_negedge")
        _add_positive(features, "has_always")

    # ------------------------------------------------------------------
    # Structural constructs
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bcase\b", r"\bopcode\b", r"\bstate\b", r"\bselect\b"
    ]):
        _add_positive(features, "has_case")

    if _contains_any(text, [
        r"\bconditional\b", r"\benable\b", r"\bif-else\b", r"\bpriority\b"
    ]):
        _add_positive(features, "has_if")

    if _contains_any(text, [
        r"\bfor loop\b", r"\biterate\b", r"\bfor each bit\b", r"\bgenerate bits\b"
    ]):
        _add_positive(features, "has_for")

    if _contains_any(text, [
        r"\bgenerate\b", r"\bparameterized instances\b", r"\bn copies\b",
        r"\binstantiates multiple\b"
    ]):
        _add_positive(features, "has_generate")

    if _contains_any(text, [
        r"\bcontinuous assignment\b", r"\bassign\b", r"\bcombinational\b",
        r"\bpure combinational\b"
    ]):
        _add_positive(features, "has_assign")


    if _contains_any(text, [
        r"\band\b", r"\bbitwise and\b"
    ]):
        _add_positive(features, "has_and_op")

    if _contains_any(text, [
        r"\bor\b", r"\bbitwise or\b"
    ]):
        _add_positive(features, "has_or_op")

    if _contains_any(text, [
        r"\bxor\b", r"\bexclusive or\b", r"\bparity\b"
    ]):
        _add_positive(features, "has_xor_op")

    if _contains_any(text, [
        r"\badd\b", r"\badder\b", r"\bsum\b"
    ]):
        _add_positive(features, "has_add_op")

    if _contains_any(text, [
        r"\bsubtract\b", r"\bsubtractor\b", r"\bdifference\b"
    ]):
        _add_positive(features, "has_sub_op")

    if _contains_any(text, [
        r"\bshift left\b", r"\bleft shift\b"
    ]):
        _add_positive(features, "has_lshift_op")

    if _contains_any(text, [
        r"\bshift right\b", r"\bright shift\b"
    ]):
        _add_positive(features, "has_rshift_op")

    if _contains_any(text, [
        r"\bconcatenate\b", r"\bconcatenation\b"
    ]):
        _add_positive(features, "has_concat")

    if _contains_any(text, [
        r"\b32 bits\b", r"\b32-bit\b", r"\b\[31:0\]\b"
    ]):
        _add_positive(features, "has_wide_port_32plus")

    if _contains_any(text, [
        r"\b8 bits\b", r"\b8-bit\b", r"\b\[7:0\]\b"
    ]):
        _add_positive(features, "has_wide_port_8plus")

    if _contains_any(text, [
        r"\boutput zero\b", r"\boutput one\b", r"\bsingle output\b"
    ]):
        _add_positive(features, "is_single_output")


    # ------------------------------------------------------------------
    # Derived implications
    # ------------------------------------------------------------------
    pos = features["positive"]

    if pos.get("tag_fsm") == 1:
        pos.setdefault("has_case", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_counter") == 1:
        pos.setdefault("has_clk_signal", 1)
        pos.setdefault("has_posedge", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_fifo") == 1:
        pos.setdefault("tag_ram", 1)
        pos.setdefault("tag_mem_array", 1)
        pos.setdefault("has_clk_signal", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_async_reset") == 1:
        pos.setdefault("has_reset_signal", 1)
        pos.setdefault("has_negedge", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_constant_output") == 1:
        pos.setdefault("is_assign_only", 1)
        pos.setdefault("is_single_output", 1)
        pos.setdefault("has_assign", 1)

    if pos.get("tag_not_gate") == 1:
        pos.setdefault("has_not_op", 1)
        pos.setdefault("is_assign_only", 1)
        pos.setdefault("has_assign", 1)

    if pos.get("tag_byte_reorder") == 1:
        pos.setdefault("has_concat", 1)
        pos.setdefault("has_wide_port_32plus", 1)
        pos.setdefault("has_assign", 1)

    if pos.get("tag_edge_detector") == 1:
        pos.setdefault("has_clk_signal", 1)
        pos.setdefault("has_posedge", 1)
        pos.setdefault("has_always", 1)

    if pos.get("has_instantiation") == 1:
        pos.setdefault("tag_wrapper_module", 1)


    # ------------------------------------------------------------------
    # Explicit negative cues
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bwithout clock\b", r"\bno clock\b", r"\bclockless\b"
    ]):
        _add_negative(features, "has_clk_signal")
        _add_negative(features, "has_posedge")
        _add_negative(features, "has_negedge")

    if _contains_any(text, [
        r"\bno reset\b", r"\bwithout reset\b"
    ]):
        _add_negative(features, "has_reset_signal")
        _add_negative(features, "tag_async_reset")

    if _contains_any(text, [
        r"\bcombinational\b", r"\bpure combinational\b"
    ]):
        _add_negative(features, "has_posedge")
        _add_negative(features, "has_negedge")
        _add_negative(features, "has_clk_signal")

    return features


def get_kg_feature_weights() -> dict:
    return {
        # original high-level tags
        "tag_fsm": 5.0,
        "tag_counter": 5.0,
        "tag_mux": 4.0,
        "tag_alu": 5.0,
        "tag_shift": 4.0,
        "tag_mem_array": 4.0,
        "tag_ram": 5.0,
        "tag_fifo": 6.0,
        "tag_async_reset": 4.0,

        # new functional tags
        "tag_constant_output": 5.0,
        "tag_not_gate": 5.0,
        "tag_encoder": 5.0,
        "tag_decoder": 5.0,
        "tag_comparator": 5.0,
        "tag_edge_detector": 5.5,
        "tag_byte_reorder": 6.0,
        "tag_wrapper_module": 4.0,

        # operator features
        "has_not_op": 3.0,
        "has_and_op": 2.0,
        "has_or_op": 2.0,
        "has_xor_op": 2.5,
        "has_add_op": 2.5,
        "has_sub_op": 2.5,
        "has_lshift_op": 2.5,
        "has_rshift_op": 2.5,
        "has_concat": 3.0,
        "has_compare_eq": 2.5,
        "has_compare_lt": 2.5,
        "has_compare_gt": 2.5,

        # sequential / clocking
        "has_posedge": 3.0,
        "has_negedge": 3.0,
        "has_clk_signal": 3.0,
        "has_reset_signal": 2.5,

        # generic structure
        "has_always": 2.0,
        "has_assign": 2.0,
        "has_case": 2.0,
        "has_if": 1.5,
        "has_for": 1.0,
        "has_generate": 1.5,
        "has_instantiation": 2.5,

        # interface / shape
        "has_wide_port_8plus": 2.0,
        "has_wide_port_32plus": 3.0,
        "is_single_output": 2.5,
        "is_assign_only": 3.0,
    }


def score_kg_row(row, positive: dict, negative: dict, weights: dict) -> float:
    score = 0.0

    for feat in positive:
        w = weights.get(feat, 1.0)
        val = int(getattr(row, feat, 0))
        if val == 1:
            score += w
        else:
            score -= 0.5 * w

    for feat in negative:
        w = weights.get(feat, 1.0)
        val = int(getattr(row, feat, 0))
        if val == 0:
            score += 0.5 * w
        else:
            score -= w

    return score


def preview(text: str, max_chars: int = MAX_PREVIEW_CHARS) -> str:
    text = text.replace("\n", " ").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "..."


def main():
    vector_df = load_vector_db()
    kg_df = load_kg_db()
    weights = get_kg_feature_weights()

    test_df = vector_df.head(NUM_TEST_INSTRUCTIONS)

    for idx, row in test_df.iterrows():
        instruction = row["Instruction"]

        query_features = extract_kg_query_features(instruction)
        positive = query_features["positive"]
        negative = query_features["negative"]

        print("=" * 100)
        print(f"QUERY ROW INDEX: {idx}")
        print("ORIGINAL INSTRUCTION:")
        print(preview(instruction, 400))
        print()

        print("EXTRACTED POSITIVE FEATURES:")
        print(positive)
        print("EXTRACTED NEGATIVE FEATURES:")
        print(negative)
        print()

        if not positive and not negative:
            print("NO KG FEATURES EXTRACTED -> retrieval not informative for this example.")
            print()
            continue

        scored = []
        for kg_row in kg_df.itertuples(index=False):
            score = score_kg_row(kg_row, positive, negative, weights)
            if score > 0:
                scored.append((int(kg_row.doc_id), float(score), kg_row.module_name))

        scored.sort(key=lambda x: x[1], reverse=True)
        top_hits = scored[:TOP_K]

        print(f"TOP {TOP_K} KG MATCHES:")
        for rank, (doc_id, score, module_name) in enumerate(top_hits, start=1):
            matched_row = vector_df.iloc[doc_id]

            print(f"\n[{rank}] doc_id={doc_id}  module_name={module_name!r}  score={score:.4f}")
            print("Instruction preview:")
            print(preview(matched_row["Instruction"]))
            print("Verilog preview:")
            print(preview(matched_row["VerilogCode"]))

        print()


if __name__ == "__main__":
    main()

<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\p'
<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\p'
C:\Users\mikel\AppData\Local\Temp\ipykernel_4632\1378951840.py:4: SyntaxWarning: invalid escape sequence '\p'
  VECTOR_PARQUET_PATH = "vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"
C:\Users\mikel\AppData\Local\Temp\ipykernel_4632\1378951840.py:5: SyntaxWarning: invalid escape sequence '\p'
  KG_PARQUET_PATH = "kg_features_parquet\part-00000-tid-7235426855037993689-380d2ae7-9517-4e28-b508-0deae127f928-125-1.c000.snappy.parquet"


QUERY ROW INDEX: 0
ORIGINAL INSTRUCTION:
You are tasked with designing a module for a simple calculator that can perform basic arithmetic operations. The module should have two inputs, `a` and `b`, and four outputs, `add`, `sub`, `mul`, and `div`. The module should be able to perform the following operations: - `add`: add `a` and `b` and output the result - `sub`: subtract `b` from `a` and output the result - `mul`: multiply `a` and `b` ...

EXTRACTED POSITIVE FEATURES:
{'has_reset_signal': 1, 'has_if': 1}
EXTRACTED NEGATIVE FEATURES:
{}

TOP 5 KG MATCHES:

[1] doc_id=0  module_name='calculator'  score=4.0000
Instruction preview:
You are tasked with designing a module for a simple calculator that can perform basic arithmetic operations. The module should have two inputs, `a` and `b`, and four outputs, `add`, `sub`, `mul`, and `div`. The module sh...
Verilog preview:
module calculator(     input logic [31:0] a,     input logic [31:0] b,     input logic reset_n,     output logic [31:0] a

In [2]:
import re
import pandas as pd

VECTOR_PARQUET_PATH = "vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"
KG_PARQUET_PATH = "kg_features_parquet\part-00000-tid-5475169348745911287-c2bd5a7c-598b-492d-a9d9-f3537b852815-141-1.c000.snappy.parquet"

NUM_TEST_INSTRUCTIONS = 5
TOP_K = 5
MAX_PREVIEW_CHARS = 220


def load_vector_db():
    df = pd.read_parquet(VECTOR_PARQUET_PATH)
    df["Instruction"] = df["Instruction"].fillna("")
    df["VerilogCode"] = df["VerilogCode"].fillna("")
    df["embedding_text"] = df["embedding_text"].fillna("")
    return df


def load_kg_db():
    df = pd.read_parquet(KG_PARQUET_PATH)

    if "module_name" in df.columns:
        df["module_name"] = df["module_name"].fillna("")

    feature_cols = [
        "has_always", "has_assign", "has_case", "has_if", "has_for", "has_generate",
        "has_posedge", "has_negedge", "has_clk_signal", "has_reset_signal",
        "tag_async_reset", "tag_fsm", "tag_counter", "tag_mux", "tag_alu",
        "tag_shift", "tag_mem_array", "tag_ram", "tag_fifo"
    ]

    for col in feature_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    return df


def _normalize_text(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"[_\-/]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _contains_any(text: str, patterns) -> bool:
    for pattern in patterns:
        if re.search(pattern, text):
            return True
    return False


def _add_positive(features: dict, name: str):
    features["positive"][name] = 1


def _add_negative(features: dict, name: str):
    features["negative"][name] = 0


def extract_kg_query_features(prompt: str) -> dict:
    """
    Rule-based prompt-to-feature extractor.
    Returns:
        {
            "positive": {feature_name: 1, ...},
            "negative": {feature_name: 0, ...}
        }
    """
    text = _normalize_text(prompt)

    features = {
        "positive": {},
        "negative": {}
    }

    if not text:
        return features

    # ------------------------------------------------------------------
    # High-confidence functional tags
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bfifo\b", r"\bqueue\b", r"\bread pointer\b", r"\bwrite pointer\b",
        r"\brd_ptr\b", r"\bwr_ptr\b", r"\bread_ptr\b", r"\bwrite_ptr\b"
    ]):
        _add_positive(features, "tag_fifo")
        _add_positive(features, "tag_ram")
        _add_positive(features, "tag_mem_array")
        _add_positive(features, "has_always")
        _add_positive(features, "has_clk_signal")

    if _contains_any(text, [
        r"\bfsm\b", r"\bfinite state machine\b", r"\bstate machine\b",
        r"\bstate transition\b", r"\bnext state\b", r"\bcurrent state\b"
    ]):
        _add_positive(features, "tag_fsm")
        _add_positive(features, "has_case")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\bcounter\b", r"\bup counter\b", r"\bdown counter\b",
        r"\bincrement\b", r"\bdecrement\b", r"\bcounts?\b"
    ]):
        _add_positive(features, "tag_counter")
        _add_positive(features, "has_always")
        _add_positive(features, "has_clk_signal")
        _add_positive(features, "has_posedge")

    if _contains_any(text, [
        r"\bmux\b", r"\bmultiplexer\b", r"\bselector\b", r"\bselect one of\b",
        r"\b4 to 1\b", r"\b2 to 1\b", r"\b8 to 1\b"
    ]):
        _add_positive(features, "tag_mux")

    if _contains_any(text, [
        r"\balu\b", r"\barithmetic logic unit\b", r"\bopcode\b",
        r"\badd/subtract\b", r"\badd and subtract\b"
    ]):
        _add_positive(features, "tag_alu")
        _add_positive(features, "has_case")

    if _contains_any(text, [
        r"\bshift register\b", r"\bshift left\b", r"\bshift right\b",
        r"\bbarrel shifter\b", r"\bshifter\b", r"\bshift\b"
    ]):
        _add_positive(features, "tag_shift")

    if _contains_any(text, [
        r"\bram\b", r"\bmemory\b", r"\bmem\b", r"\bstorage array\b"
    ]):
        _add_positive(features, "tag_ram")
        _add_positive(features, "tag_mem_array")

    if _contains_any(text, [
        r"\balways output[s]?\s+(a\s+)?low\b",
        r"\balways output[s]?\s+(a\s+)?high\b",
        r"\balways drive[s]?\s+0\b",
        r"\balways drive[s]?\s+1\b",
        r"\balways drive[s]?\s+low\b",
        r"\balways drive[s]?\s+high\b",
        r"\bconstant output\b",
        r"\bconstant zero\b",
        r"\bconstant one\b"
    ]):
        _add_positive(features, "tag_constant_output")
        _add_positive(features, "is_assign_only")
        _add_positive(features, "is_single_output")

    if _contains_any(text, [
        r"\bnot gate\b", r"\binverter\b", r"\binvert\b", r"\bbitwise not\b"
    ]):
        _add_positive(features, "tag_not_gate")
        _add_positive(features, "has_not_op")
        _add_positive(features, "is_assign_only")

    if _contains_any(text, [
        r"\bencoder\b", r"\bpriority encoder\b"
    ]):
        _add_positive(features, "tag_encoder")
        _add_positive(features, "has_if")

    if _contains_any(text, [
        r"\bdecoder\b"
    ]):
        _add_positive(features, "tag_decoder")

    if _contains_any(text, [
        r"\bcomparator\b", r"\bcompare\b", r"\bequal\b", r"\bgreater than\b",
        r"\bless than\b", r"\bgreater-than\b", r"\bless-than\b"
    ]):
        _add_positive(features, "tag_comparator")

    if _contains_any(text, [
        r"\bedge detector\b", r"\bdetect any edge\b", r"\brising edge detector\b",
        r"\bfalling edge detector\b", r"\bedge detect\b"
    ]):
        _add_positive(features, "tag_edge_detector")
        _add_positive(features, "has_clk_signal")
        _add_positive(features, "has_posedge")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\breverse the byte order\b", r"\bbyte order\b", r"\bbyte reorder\b",
        r"\bbyte swap\b", r"\bendian\b"
    ]):
        _add_positive(features, "tag_byte_reorder")
        _add_positive(features, "has_concat")
        _add_positive(features, "has_wide_port_32plus")

    if _contains_any(text, [
        r"\binstantiate\b", r"\binstantiates\b", r"\btop level module\b",
        r"\btop-level module\b", r"\bwrapper\b", r"\bsubmodule\b"
    ]):
        _add_positive(features, "has_instantiation")

    # ------------------------------------------------------------------
    # Clock/reset/sequential style
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bclock\b", r"\bclk\b", r"\bsynchronous\b", r"\bposedge\b",
        r"\brising edge\b"
    ]):
        _add_positive(features, "has_clk_signal")
        _add_positive(features, "has_posedge")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\bnegedge\b", r"\bfalling edge\b"
    ]):
        _add_positive(features, "has_negedge")
        _add_positive(features, "has_always")

    if _contains_any(text, [
        r"\breset\b", r"\brst\b"
    ]):
        _add_positive(features, "has_reset_signal")

    if _contains_any(text, [
        r"\basynchronous reset\b", r"\basync reset\b", r"\bactive low reset\b",
        r"\bactive-low reset\b", r"\bnegedge reset\b", r"\bnegedge rst\b"
    ]):
        _add_positive(features, "tag_async_reset")
        _add_positive(features, "has_reset_signal")
        _add_positive(features, "has_negedge")
        _add_positive(features, "has_always")

    # ------------------------------------------------------------------
    # Structural constructs
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bcase\b", r"\bopcode\b", r"\bstate\b", r"\bselect\b"
    ]):
        _add_positive(features, "has_case")

    if _contains_any(text, [
        r"\bconditional\b", r"\benable\b", r"\bif-else\b", r"\bpriority\b"
    ]):
        _add_positive(features, "has_if")

    if _contains_any(text, [
        r"\bfor loop\b", r"\biterate\b", r"\bfor each bit\b", r"\bgenerate bits\b"
    ]):
        _add_positive(features, "has_for")

    if _contains_any(text, [
        r"\bgenerate\b", r"\bparameterized instances\b", r"\bn copies\b",
        r"\binstantiates multiple\b"
    ]):
        _add_positive(features, "has_generate")

    if _contains_any(text, [
        r"\bcontinuous assignment\b", r"\bassign\b", r"\bcombinational\b",
        r"\bpure combinational\b"
    ]):
        _add_positive(features, "has_assign")


    if _contains_any(text, [
        r"\band\b", r"\bbitwise and\b"
    ]):
        _add_positive(features, "has_and_op")

    if _contains_any(text, [
        r"\bor\b", r"\bbitwise or\b"
    ]):
        _add_positive(features, "has_or_op")

    if _contains_any(text, [
        r"\bxor\b", r"\bexclusive or\b", r"\bparity\b"
    ]):
        _add_positive(features, "has_xor_op")

    if _contains_any(text, [
        r"\badd\b", r"\badder\b", r"\bsum\b"
    ]):
        _add_positive(features, "has_add_op")

    if _contains_any(text, [
        r"\bsubtract\b", r"\bsubtractor\b", r"\bdifference\b"
    ]):
        _add_positive(features, "has_sub_op")

    if _contains_any(text, [
        r"\bshift left\b", r"\bleft shift\b"
    ]):
        _add_positive(features, "has_lshift_op")

    if _contains_any(text, [
        r"\bshift right\b", r"\bright shift\b"
    ]):
        _add_positive(features, "has_rshift_op")

    if _contains_any(text, [
        r"\bconcatenate\b", r"\bconcatenation\b"
    ]):
        _add_positive(features, "has_concat")

    if _contains_any(text, [
        r"\b32 bits\b", r"\b32-bit\b", r"\b\[31:0\]\b"
    ]):
        _add_positive(features, "has_wide_port_32plus")

    if _contains_any(text, [
        r"\b8 bits\b", r"\b8-bit\b", r"\b\[7:0\]\b"
    ]):
        _add_positive(features, "has_wide_port_8plus")

    if _contains_any(text, [
        r"\boutput zero\b", r"\boutput one\b", r"\bsingle output\b"
    ]):
        _add_positive(features, "is_single_output")


    # ------------------------------------------------------------------
    # Derived implications
    # ------------------------------------------------------------------
    pos = features["positive"]

    if pos.get("tag_fsm") == 1:
        pos.setdefault("has_case", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_counter") == 1:
        pos.setdefault("has_clk_signal", 1)
        pos.setdefault("has_posedge", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_fifo") == 1:
        pos.setdefault("tag_ram", 1)
        pos.setdefault("tag_mem_array", 1)
        pos.setdefault("has_clk_signal", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_async_reset") == 1:
        pos.setdefault("has_reset_signal", 1)
        pos.setdefault("has_negedge", 1)
        pos.setdefault("has_always", 1)

    if pos.get("tag_constant_output") == 1:
        pos.setdefault("is_assign_only", 1)
        pos.setdefault("is_single_output", 1)
        pos.setdefault("has_assign", 1)

    if pos.get("tag_not_gate") == 1:
        pos.setdefault("has_not_op", 1)
        pos.setdefault("is_assign_only", 1)
        pos.setdefault("has_assign", 1)

    if pos.get("tag_byte_reorder") == 1:
        pos.setdefault("has_concat", 1)
        pos.setdefault("has_wide_port_32plus", 1)
        pos.setdefault("has_assign", 1)

    if pos.get("tag_edge_detector") == 1:
        pos.setdefault("has_clk_signal", 1)
        pos.setdefault("has_posedge", 1)
        pos.setdefault("has_always", 1)

    if pos.get("has_instantiation") == 1:
        pos.setdefault("tag_wrapper_module", 1)


    # ------------------------------------------------------------------
    # Explicit negative cues
    # ------------------------------------------------------------------
    if _contains_any(text, [
        r"\bwithout clock\b", r"\bno clock\b", r"\bclockless\b"
    ]):
        _add_negative(features, "has_clk_signal")
        _add_negative(features, "has_posedge")
        _add_negative(features, "has_negedge")

    if _contains_any(text, [
        r"\bno reset\b", r"\bwithout reset\b"
    ]):
        _add_negative(features, "has_reset_signal")
        _add_negative(features, "tag_async_reset")

    if _contains_any(text, [
        r"\bcombinational\b", r"\bpure combinational\b"
    ]):
        _add_negative(features, "has_posedge")
        _add_negative(features, "has_negedge")
        _add_negative(features, "has_clk_signal")

    return features


def get_kg_feature_weights() -> dict:
    return {
        # original high-level tags
        "tag_fsm": 5.0,
        "tag_counter": 5.0,
        "tag_mux": 4.0,
        "tag_alu": 5.0,
        "tag_shift": 4.0,
        "tag_mem_array": 4.0,
        "tag_ram": 5.0,
        "tag_fifo": 6.0,
        "tag_async_reset": 4.0,

        # new functional tags
        "tag_constant_output": 5.0,
        "tag_not_gate": 5.0,
        "tag_encoder": 5.0,
        "tag_decoder": 5.0,
        "tag_comparator": 5.0,
        "tag_edge_detector": 5.5,
        "tag_byte_reorder": 6.0,
        "tag_wrapper_module": 4.0,

        # operator features
        "has_not_op": 3.0,
        "has_and_op": 2.0,
        "has_or_op": 2.0,
        "has_xor_op": 2.5,
        "has_add_op": 2.5,
        "has_sub_op": 2.5,
        "has_lshift_op": 2.5,
        "has_rshift_op": 2.5,
        "has_concat": 3.0,
        "has_compare_eq": 2.5,
        "has_compare_lt": 2.5,
        "has_compare_gt": 2.5,

        # sequential / clocking
        "has_posedge": 3.0,
        "has_negedge": 3.0,
        "has_clk_signal": 3.0,
        "has_reset_signal": 2.5,

        # generic structure
        "has_always": 2.0,
        "has_assign": 2.0,
        "has_case": 2.0,
        "has_if": 1.5,
        "has_for": 1.0,
        "has_generate": 1.5,
        "has_instantiation": 2.5,

        # interface / shape
        "has_wide_port_8plus": 2.0,
        "has_wide_port_32plus": 3.0,
        "is_single_output": 2.5,
        "is_assign_only": 3.0,
    }


def score_kg_row(row, positive: dict, negative: dict, weights: dict) -> float:
    score = 0.0

    for feat in positive:
        w = weights.get(feat, 1.0)
        val = int(getattr(row, feat, 0))
        if val == 1:
            score += w
        else:
            score -= 0.5 * w

    for feat in negative:
        w = weights.get(feat, 1.0)
        val = int(getattr(row, feat, 0))
        if val == 0:
            score += 0.5 * w
        else:
            score -= w

    return score


def preview(text: str, max_chars: int = MAX_PREVIEW_CHARS) -> str:
    text = text.replace("\n", " ").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "..."


def main():
    vector_df = load_vector_db()
    kg_df = load_kg_db()
    weights = get_kg_feature_weights()

    test_df = vector_df.head(NUM_TEST_INSTRUCTIONS)

    for idx, row in test_df.iterrows():
        instruction = row["Instruction"]

        query_features = extract_kg_query_features(instruction)
        positive = query_features["positive"]
        negative = query_features["negative"]

        print("=" * 100)
        print(f"QUERY ROW INDEX: {idx}")
        print("ORIGINAL INSTRUCTION:")
        print(preview(instruction, 400))
        print()

        print("EXTRACTED POSITIVE FEATURES:")
        print(positive)
        print("EXTRACTED NEGATIVE FEATURES:")
        print(negative)
        print()

        if not positive and not negative:
            print("NO KG FEATURES EXTRACTED -> retrieval not informative for this example.")
            print()
            continue

        scored = []
        for kg_row in kg_df.itertuples(index=False):
            score = score_kg_row(kg_row, positive, negative, weights)
            if score > 0:
                scored.append((int(kg_row.doc_id), float(score), kg_row.module_name))

        scored.sort(key=lambda x: x[1], reverse=True)
        top_hits = scored[:TOP_K]

        print(f"TOP {TOP_K} KG MATCHES:")
        for rank, (doc_id, score, module_name) in enumerate(top_hits, start=1):
            matched_row = vector_df.iloc[doc_id]

            print(f"\n[{rank}] doc_id={doc_id}  module_name={module_name!r}  score={score:.4f}")
            print("Instruction preview:")
            print(preview(matched_row["Instruction"]))
            print("Verilog preview:")
            print(preview(matched_row["VerilogCode"]))

        print()


if __name__ == "__main__":
    main()

<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\p'
<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:5: SyntaxWarning: invalid escape sequence '\p'
C:\Users\mikel\AppData\Local\Temp\ipykernel_25968\1940741511.py:4: SyntaxWarning: invalid escape sequence '\p'
  VECTOR_PARQUET_PATH = "vector_parquet\part-00000-tid-5189121977065850180-3b867b6d-c7f2-4576-b96b-f7dd87bf9dfd-126-1.c000.snappy.parquet"
C:\Users\mikel\AppData\Local\Temp\ipykernel_25968\1940741511.py:5: SyntaxWarning: invalid escape sequence '\p'
  KG_PARQUET_PATH = "kg_features_parquet\part-00000-tid-5475169348745911287-c2bd5a7c-598b-492d-a9d9-f3537b852815-141-1.c000.snappy.parquet"


QUERY ROW INDEX: 0
ORIGINAL INSTRUCTION:
You are tasked with designing a module for a simple calculator that can perform basic arithmetic operations. The module should have two inputs, `a` and `b`, and four outputs, `add`, `sub`, `mul`, and `div`. The module should be able to perform the following operations: - `add`: add `a` and `b` and output the result - `sub`: subtract `b` from `a` and output the result - `mul`: multiply `a` and `b` ...

EXTRACTED POSITIVE FEATURES:
{'has_reset_signal': 1, 'has_and_op': 1, 'has_add_op': 1, 'has_sub_op': 1}
EXTRACTED NEGATIVE FEATURES:
{}

TOP 5 KG MATCHES:

[1] doc_id=25  module_name='fifo_async_4bit_wr_logic'  score=9.5000
Instruction preview:
You are tasked with designing a verilog module that implements a 4-bit asynchronous FIFO write logic. The module should have the following inputs and outputs:  Inputs: - `wr_clk`: a clock input for the write logic - `rst...
Verilog preview:
module fifo_async_4bit_wr_logic    (full,     prog_full,     Q,     

## VeriRAG KG Construction and Verification

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

# =========================================================
# Paths
# =========================================================
file_path = Path("vector_parquet/verirag_clean.parquet")
output_path = Path("kg_features_parquet/verirag_kg_features.parquet")

# =========================================================
# Load dataset
# =========================================================
df = pd.read_parquet(file_path).copy()

print("Input shape:", df.shape)
print(df.columns.tolist())
print(df.head(3))

# =========================================================
# Base columns
# =========================================================
df = df.reset_index(drop=True).copy()
df["doc_id"] = df.index

df["Instruction"] = df["Instruction"].fillna("")
df["VerilogCode"] = df["VerilogCode"].fillna("")

df["code_lc"] = df["VerilogCode"].str.lower()

def strip_comments(code: str) -> str:
    if not isinstance(code, str):
        return ""
    code = re.sub(r"(?s)/\*.*?\*/", " ", code)   # block comments
    code = re.sub(r"//.*", " ", code)           # line comments
    return code

df["code_no_comments"] = df["code_lc"].apply(strip_comments)

def extract_module_name(code: str) -> str:
    if not isinstance(code, str):
        return ""
    m = re.search(r"(?i)\bmodule\s+([a-zA-Z_]\w*)", code)
    return m.group(1) if m else ""

df["module_name"] = df["VerilogCode"].apply(extract_module_name)

# =========================================================
# Helper functions
# =========================================================
def contains_substr(text: str, substr: str) -> int:
    return int(substr in text) if isinstance(text, str) else 0

def contains_regex(text: str, pattern: str, flags=0) -> int:
    if not isinstance(text, str):
        return 0
    return int(re.search(pattern, text, flags) is not None)

def count_keyword(text: str, keyword: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(rf"(?i)\b{re.escape(keyword)}\b", text))

def max_decl_width(code: str) -> int:
    """
    Approximate max declared width from [N:0] or [0:N]
    Returns 1 when no vector declaration is found.
    """
    if not isinstance(code, str):
        return 1

    widths = [1]

    for a, b in re.findall(r"\[\s*(\d+)\s*:\s*(\d+)\s*\]", code):
        try:
            widths.append(abs(int(a) - int(b)) + 1)
        except ValueError:
            pass

    return max(widths)

# =========================================================
# Basic structural features
# =========================================================
df["has_always"]   = df["code_no_comments"].apply(lambda x: contains_substr(x, "always"))
df["has_assign"]   = df["code_no_comments"].apply(lambda x: contains_substr(x, "assign"))
df["has_case"]     = df["code_no_comments"].apply(lambda x: contains_substr(x, "case"))
df["has_if"]       = df["code_no_comments"].apply(lambda x: contains_substr(x, "if"))
df["has_for"]      = df["code_no_comments"].apply(lambda x: contains_substr(x, "for"))
df["has_generate"] = df["code_no_comments"].apply(lambda x: contains_substr(x, "generate"))
df["has_posedge"]  = df["code_no_comments"].apply(lambda x: contains_substr(x, "posedge"))
df["has_negedge"]  = df["code_no_comments"].apply(lambda x: contains_substr(x, "negedge"))

# =========================================================
# Low-level operator / expression features
# =========================================================
df["has_not_op"]     = df["code_no_comments"].apply(lambda x: int("~"  in x))
df["has_and_op"]     = df["code_no_comments"].apply(lambda x: int(("&" in x) or ("&&" in x)))
df["has_or_op"]      = df["code_no_comments"].apply(lambda x: int(("|" in x) or ("||" in x)))
df["has_xor_op"]     = df["code_no_comments"].apply(lambda x: int("^"  in x))
df["has_add_op"]     = df["code_no_comments"].apply(lambda x: int("+"  in x))
df["has_sub_op"]     = df["code_no_comments"].apply(lambda x: int("-"  in x))
df["has_lshift_op"]  = df["code_no_comments"].apply(lambda x: int("<<" in x))
df["has_rshift_op"]  = df["code_no_comments"].apply(lambda x: int(">>" in x))
df["has_concat"]     = df["code_no_comments"].apply(lambda x: contains_regex(x, r"\{[^{}]+\}"))
df["has_compare_eq"] = df["code_no_comments"].apply(lambda x: int(("===" in x) or ("==" in x)))
df["has_compare_lt"] = df["code_no_comments"].apply(lambda x: int("<" in x))
df["has_compare_gt"] = df["code_no_comments"].apply(lambda x: int(">" in x))

# =========================================================
# Coarse tags
# =========================================================
df["tag_fsm"] = df["code_no_comments"].apply(
    lambda x: int(("state" in x) and ("case" in x))
)

df["tag_counter"] = df["code_no_comments"].apply(
    lambda x: int(("+ 1" in x) or ("++" in x) or ("counter" in x))
)

df["tag_mux"] = df["code_no_comments"].apply(
    lambda x: int(("case" in x) or ("?:" in x))
)

df["tag_alu"] = (
    ((df["code_no_comments"].str.contains("alu", regex=False)).astype(int) == 1) |
    ((df["has_add_op"] == 1) & (df["has_sub_op"] == 1))
).astype(int)

df["tag_shift"] = (
    (df["has_lshift_op"] == 1) |
    (df["has_rshift_op"] == 1) |
    (df["code_no_comments"].str.contains("shift", regex=False)).fillna(False)
).astype(int)

# =========================================================
# Memory-related features
# =========================================================
df["tag_mem_array"] = df["VerilogCode"].apply(
    lambda x: contains_regex(
        x,
        r"(?is)\breg\b[^\n;]*\[[^\]]+\]\s*\w+\s*\[[^\]]+\]\s*;"
    )
)

df["tag_ram"] = (
    (df["code_no_comments"].str.contains("ram", regex=False)).fillna(False) |
    (df["code_no_comments"].str.contains("memory", regex=False)).fillna(False) |
    (df["tag_mem_array"] == 1)
).astype(int)

df["tag_fifo"] = df["code_no_comments"].apply(
    lambda x: int(
        ("fifo" in x) or
        (("wr_ptr" in x) and ("rd_ptr" in x)) or
        (("write_ptr" in x) and ("read_ptr" in x))
    )
)

# =========================================================
# Clock / reset / sequential style
# =========================================================
df["has_clk_signal"] = df["code_no_comments"].apply(
    lambda x: int(
        (" clk" in x) or
        ("(clk" in x) or
        ("clock" in x)
    )
)

df["has_reset_signal"] = df["code_no_comments"].apply(
    lambda x: int(("reset" in x) or ("rst" in x))
)

df["tag_async_reset"] = df["code_no_comments"].apply(
    lambda x: int(
        ("negedge rst" in x) or
        ("negedge reset" in x) or
        ("posedge reset" in x) or
        ("or negedge" in x) or
        ("or posedge reset" in x)
    )
)

# =========================================================
# Functional tags
# =========================================================
df["tag_constant_output"] = df["code_no_comments"].apply(
    lambda x: int(
        re.search(r"assign\s+\w+\s*=\s*(1'b0|1'b1|0|1)\s*;", x) is not None or
        re.search(r"\w+\s*<=\s*(1'b0|1'b1|0|1)\s*;", x) is not None
    )
)

df["tag_not_gate"] = ((df["has_not_op"] == 1) & (df["has_assign"] == 1)).astype(int)

df["tag_encoder"] = df["code_no_comments"].apply(
    lambda x: int(
        (("encoder" in x) or ("priority" in x)) and
        (("if" in x) or ("case" in x))
    )
)

df["tag_decoder"] = df["code_no_comments"].apply(
    lambda x: int("decoder" in x)
)

df["tag_comparator"] = (
    (df["code_no_comments"].str.contains("compare", regex=False)).fillna(False) |
    (df["has_compare_eq"] == 1) |
    (df["has_compare_lt"] == 1) |
    (df["has_compare_gt"] == 1)
).astype(int)

df["tag_edge_detector"] = df["code_no_comments"].apply(
    lambda x: int(
        ("edge" in x) or
        (("prev_" in x) and ((" clk" in x) or ("(clk" in x) or ("clock" in x))) or
        (("prev" in x) and ("^" in x))
    )
)

df["tag_byte_reorder"] = df["code_no_comments"].apply(
    lambda x: contains_regex(
        x,
        r"\{\s*\w+\[\d+:\d+\]\s*,\s*\w+\[\d+:\d+\]\s*,\s*\w+\[\d+:\d+\]\s*,\s*\w+\[\d+:\d+\]\s*\}"
    )
)

# =========================================================
# Instantiation / wrapper style
# =========================================================
df["has_instantiation"] = df["VerilogCode"].apply(
    lambda x: contains_regex(
        x,
        r"(?im)^\s*[A-Za-z_]\w*\s+[A-Za-z_]\w*\s*\("
    )
)

df["tag_wrapper_module"] = (
    (df["has_instantiation"] == 1) &
    (df["has_assign"] == 0) &
    (df["has_always"] == 0)
).astype(int)

# =========================================================
# Interface / declaration shape features
# =========================================================
df["num_inputs"] = df["VerilogCode"].apply(lambda x: count_keyword(x, "input"))
df["num_outputs"] = df["VerilogCode"].apply(lambda x: count_keyword(x, "output"))
df["max_decl_width"] = df["VerilogCode"].apply(max_decl_width)

df["has_wide_port_8plus"] = (df["max_decl_width"] >= 8).astype(int)
df["has_wide_port_32plus"] = (df["max_decl_width"] >= 32).astype(int)
df["is_single_output"] = (df["num_outputs"] == 1).astype(int)

# =========================================================
# Assign-only / simple combinational shape
# =========================================================
df["is_assign_only"] = (
    (df["has_assign"] == 1) &
    (df["has_always"] == 0) &
    (df["has_instantiation"] == 0)
).astype(int)

# =========================================================
# Final KG table
# =========================================================
kg_table = df[[
    "doc_id",
    "module_name",

    # original structural features
    "has_always", "has_assign", "has_case", "has_if", "has_for", "has_generate",
    "has_posedge", "has_negedge",
    "has_clk_signal", "has_reset_signal", "tag_async_reset",
    "tag_fsm", "tag_counter", "tag_mux", "tag_alu", "tag_shift",
    "tag_mem_array", "tag_ram", "tag_fifo",

    # operator features
    "has_not_op", "has_and_op", "has_or_op", "has_xor_op",
    "has_add_op", "has_sub_op", "has_lshift_op", "has_rshift_op",
    "has_concat", "has_compare_eq", "has_compare_lt", "has_compare_gt",

    # functional tags
    "tag_constant_output", "tag_not_gate", "tag_encoder", "tag_decoder",
    "tag_comparator", "tag_edge_detector", "tag_byte_reorder",
    "has_instantiation", "tag_wrapper_module",

    # interface / shape
    "num_inputs", "num_outputs", "max_decl_width",
    "has_wide_port_8plus", "has_wide_port_32plus", "is_single_output",
    "is_assign_only"
]].copy()

# force integer dtype on feature columns
for col in kg_table.columns:
    if col not in ["module_name"]:
        kg_table[col] = kg_table[col].fillna(0).astype(int)

print("\nKG shape:", kg_table.shape)
print(kg_table.head(10))

# =========================================================
# Save parquet
# =========================================================
output_path.parent.mkdir(parents=True, exist_ok=True)
kg_table.to_parquet(output_path, index=False)

print(f"\nSaved KG features to: {output_path}")

Input shape: (161, 4)
['Instruction', 'VerilogCode', 'id', 'embedding_text']
                                  Instruction  \
0  Write Verilog code for a 2-input AND gate.   
1   Write Verilog code for a 2-input OR gate.   
2          Write Verilog code for a NOT gate.   

                                         VerilogCode  id  \
0  module and_gate (input wire a, input wire b, o...   0   
1  module or_gate (input wire a, input wire b, ou...   1   
2  module not_gate (input wire a, output wire y);...   2   

                               embedding_text  
0  Write Verilog code for a 2-input AND gate.  
1   Write Verilog code for a 2-input OR gate.  
2          Write Verilog code for a NOT gate.  

KG shape: (161, 49)
   doc_id         module_name  has_always  has_assign  has_case  has_if  \
0       0            and_gate           0           1         0       0   
1       1             or_gate           0           1         0       0   
2       2            not_gate           0      

In [2]:
feature_cols = [c for c in kg_table.columns if c not in ["doc_id", "module_name"]]

summary = pd.DataFrame({
    "feature": feature_cols,
    "sum": [kg_table[c].sum() for c in feature_cols],
    "mean": [kg_table[c].mean() for c in feature_cols],
})

print(summary.sort_values("sum", ascending=False).head(20))

                feature   sum      mean
42       max_decl_width  1328  8.248447
40           num_inputs   392  2.434783
41          num_outputs   291  1.807453
0            has_always   126  0.782609
35       tag_comparator   120  0.745342
29       has_compare_lt   117  0.726708
3                has_if   116  0.720497
45     is_single_output   110  0.683230
38    has_instantiation   110  0.683230
36    tag_edge_detector   101  0.627329
6           has_posedge   101  0.627329
8        has_clk_signal   100  0.621118
9      has_reset_signal    90  0.559006
17              tag_ram    83  0.515528
10      tag_async_reset    82  0.509317
24           has_sub_op    75  0.465839
31  tag_constant_output    73  0.453416
23           has_add_op    59  0.366460
12          tag_counter    57  0.354037
43  has_wide_port_8plus    49  0.304348


In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# Paths
# =========================================================
vector_path = Path("vector_parquet/verirag_clean.parquet")
kg_path = Path("kg_features_parquet/verirag_kg_features.parquet")

# =========================================================
# Load files
# =========================================================
vector_df = pd.read_parquet(vector_path).copy()
kg_df = pd.read_parquet(kg_path).copy()

print("Vector shape:", vector_df.shape)
print("KG shape:", kg_df.shape)

# =========================================================
# Basic schema validation
# =========================================================
required_kg_cols = [
    "doc_id", "module_name",
    "has_always", "has_assign", "has_case", "has_if", "has_for", "has_generate",
    "has_posedge", "has_negedge",
    "has_clk_signal", "has_reset_signal", "tag_async_reset",
    "tag_fsm", "tag_counter", "tag_mux", "tag_alu", "tag_shift",
    "tag_mem_array", "tag_ram", "tag_fifo",
    "has_not_op", "has_and_op", "has_or_op", "has_xor_op",
    "has_add_op", "has_sub_op", "has_lshift_op", "has_rshift_op",
    "has_concat", "has_compare_eq", "has_compare_lt", "has_compare_gt",
    "tag_constant_output", "tag_not_gate", "tag_encoder", "tag_decoder",
    "tag_comparator", "tag_edge_detector", "tag_byte_reorder",
    "has_instantiation", "tag_wrapper_module",
    "num_inputs", "num_outputs", "max_decl_width",
    "has_wide_port_8plus", "has_wide_port_32plus", "is_single_output",
    "is_assign_only"
]

missing_cols = [c for c in required_kg_cols if c not in kg_df.columns]
extra_cols = [c for c in kg_df.columns if c not in required_kg_cols]

print("\nMissing KG columns:", missing_cols)
print("Extra KG columns:", extra_cols)

# =========================================================
# Row alignment validation
# =========================================================
if "doc_id" not in kg_df.columns:
    raise ValueError("KG file does not contain doc_id")

if len(vector_df) != len(kg_df):
    print("\nWARNING: vector and KG row counts differ")
else:
    print("\nRow counts match")

expected_doc_ids = set(range(len(vector_df)))
actual_doc_ids = set(kg_df["doc_id"].tolist())

missing_doc_ids = sorted(expected_doc_ids - actual_doc_ids)
extra_doc_ids_found = sorted(actual_doc_ids - expected_doc_ids)

print("Missing doc_ids:", missing_doc_ids[:20], "..." if len(missing_doc_ids) > 20 else "")
print("Extra doc_ids:", extra_doc_ids_found[:20], "..." if len(extra_doc_ids_found) > 20 else "")

if kg_df["doc_id"].duplicated().any():
    dupes = kg_df.loc[kg_df["doc_id"].duplicated(), "doc_id"].tolist()
    print("Duplicate doc_ids found:", dupes[:20])

# =========================================================
# Merge for row-by-row inspection
# =========================================================
merged = vector_df.copy()
merged["doc_id"] = merged.index
merged = merged.merge(kg_df, on="doc_id", how="left", suffixes=("", "_kg"))

print("\nMerged shape:", merged.shape)

# =========================================================
# Human-readable validation helper
# =========================================================
feature_cols = [c for c in required_kg_cols if c not in ["doc_id", "module_name"]]

def show_row(doc_id: int):
    row = merged.loc[merged["doc_id"] == doc_id]
    if row.empty:
        print(f"\nRow {doc_id} not found")
        return

    row = row.iloc[0]

    print("=" * 100)
    print(f"doc_id: {row['doc_id']}")
    print(f"module_name: {row['module_name']}")
    print("\nInstruction:")
    print(row["Instruction"])
    print("\nVerilogCode:")
    print(row["VerilogCode"])

    print("\nActive KG features:")
    active = []
    for c in feature_cols:
        val = row[c]
        if isinstance(val, (int, float)) and val != 0:
            active.append((c, val))

    if active:
        for name, val in active:
            print(f"  {name}: {val}")
    else:
        print("  None")

    print("=" * 100)

# =========================================================
# Quick summary of feature activation
# =========================================================
summary_rows = []
for c in feature_cols:
    summary_rows.append({
        "feature": c,
        "nonzero_count": int((merged[c] != 0).sum()),
        "mean": float(merged[c].mean())
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["nonzero_count", "feature"],
    ascending=[False, True]
)

print("\nTop activated features:")
print(summary_df.head(25).to_string(index=False))

# =========================================================
# Heuristic anomaly checks
# =========================================================
anomalies = []

for _, row in merged.iterrows():
    doc_id = int(row["doc_id"])
    instr = str(row["Instruction"]).lower()
    code = str(row["VerilogCode"]).lower()

    # 1) assign-only row should usually have no always
    if row["is_assign_only"] == 1 and row["has_always"] == 1:
        anomalies.append((doc_id, "is_assign_only=1 but has_always=1"))

    # 2) NOT gate instruction but no not feature
    if "not gate" in instr and row["tag_not_gate"] == 0:
        anomalies.append((doc_id, "Instruction says NOT gate but tag_not_gate=0"))

    # 3) AND gate instruction but no and operator
    if "and gate" in instr and row["has_and_op"] == 0:
        anomalies.append((doc_id, "Instruction says AND gate but has_and_op=0"))

    # 4) OR gate instruction but no or operator
    if "or gate" in instr and row["has_or_op"] == 0:
        anomalies.append((doc_id, "Instruction says OR gate but has_or_op=0"))

    # 5) XOR gate instruction but no xor operator
    if "xor gate" in instr and row["has_xor_op"] == 0:
        anomalies.append((doc_id, "Instruction says XOR gate but has_xor_op=0"))

    # 6) module_name missing
    if not str(row["module_name"]).strip():
        anomalies.append((doc_id, "module_name is empty"))

    # 7) one-output expectation for gate-like modules
    if "gate" in instr and row["is_single_output"] == 0:
        anomalies.append((doc_id, "Gate-like instruction but is_single_output=0"))

    # 8) assign in code but has_assign = 0
    if "assign" in code and row["has_assign"] == 0:
        anomalies.append((doc_id, "Code contains 'assign' but has_assign=0"))

    # 9) always in code but has_always = 0
    if "always" in code and row["has_always"] == 0:
        anomalies.append((doc_id, "Code contains 'always' but has_always=0"))

    # 10) posedge in code but has_posedge = 0
    if "posedge" in code and row["has_posedge"] == 0:
        anomalies.append((doc_id, "Code contains 'posedge' but has_posedge=0"))

anomalies_df = pd.DataFrame(anomalies, columns=["doc_id", "issue"]).drop_duplicates()

print("\nAnomalies found:", len(anomalies_df))
if not anomalies_df.empty:
    print(anomalies_df.head(30).to_string(index=False))

# =========================================================
# Save anomaly report
# =========================================================
report_path = Path("kg_features_parquet/verirag_kg_validation_report.csv")
anomalies_df.to_csv(report_path, index=False)
print(f"\nSaved anomaly report to: {report_path}")



Vector shape: (161, 4)
KG shape: (161, 49)

Missing KG columns: []
Extra KG columns: []

Row counts match
Missing doc_ids: [] 
Extra doc_ids: [] 

Merged shape: (161, 53)

Top activated features:
            feature  nonzero_count     mean
     max_decl_width            161 8.248447
        num_outputs            160 1.807453
         num_inputs            158 2.434783
         has_always            126 0.782609
     tag_comparator            120 0.745342
     has_compare_lt            117 0.726708
             has_if            116 0.720497
  has_instantiation            110 0.683230
   is_single_output            110 0.683230
        has_posedge            101 0.627329
  tag_edge_detector            101 0.627329
     has_clk_signal            100 0.621118
   has_reset_signal             90 0.559006
            tag_ram             83 0.515528
    tag_async_reset             82 0.509317
         has_sub_op             75 0.465839
tag_constant_output             73 0.453416
         has